In [66]:
import torch
import torch.nn as nn


In [ ]:
FILE_PATH = ""
BLOCK_SIZE = 64

In [68]:
with open(FILE_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print("Length:", len(text),'\n')
print(text[:500])

Length: 17313 

May 2026

How do you convert between wealth and income tax? If a government imposes a wealth tax of 1%, what's the equivalent in income tax?

It's clear from the way most politicians talk about the subject that they not only don't know the answer, but don't even realize there's such a question.

In fact the conversion rate between them is about 20. A wealth tax of 1% is equivalent to an income tax of 20%.

To convert between wealth and income tax rates, you have to divide by the rate of return o


In [69]:
chars = sorted(list(set(text)))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(chars)

print("Vocabulary Size:", vocab_size,'\n')
print(chars)

Vocabulary Size: 76 

['\n', ' ', '"', '$', '%', "'", '(', ')', '+', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '9', ':', ';', '=', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'W', 'Y', '[', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—']


In [70]:
data = [stoi[ch] for ch in text]

print(data[:50])

print(len(data))

[37, 49, 73, 1, 14, 12, 14, 18, 0, 0, 32, 63, 71, 1, 52, 63, 1, 73, 63, 69, 1, 51, 63, 62, 70, 53, 66, 68, 1, 50, 53, 68, 71, 53, 53, 62, 1, 71, 53, 49, 60, 68, 56, 1, 49, 62, 52, 1, 57, 62]
17313


In [71]:
xs = []
ys = []

for i in range(len(data) - BLOCK_SIZE):

    x = data[i:i + BLOCK_SIZE]

    y = data[i + 1:i + BLOCK_SIZE + 1]

    xs.append(x)
    ys.append(y)

print(len(xs))

17249


In [ ]:
X = torch.tensor(xs, dtype=torch.long)
Y = torch.tensor(ys, dtype=torch.long)

print(X.shape)
print(Y.shape)

torch.Size([17249, 64])
torch.Size([17249, 64])


In [75]:
def generate(
    model,
    device,
    start_text,
    length=500,
):

    model.eval()

    ids = [
        stoi[ch]
        for ch in start_text
    ]

    with torch.no_grad():

        for _ in range(length):

            x = torch.tensor(
                [ids],
                dtype=torch.long,
                device=device
            )

            logits = model(x)

            last = logits[-1,0]

            probs = torch.softmax(
                last,
                dim=0
            )

            next_id = torch.multinomial(
                probs,
                1
            ).item()

            ids.append(next_id)

    return "".join(
        itos[i]
        for i in ids
    )

In [76]:
class LSTM_Torch(nn.Module):

    def __init__(self,vocab_size,embedding_dim,hidden_size):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size,embedding_dim)

        self.hidden_size = hidden_size

        concat_size = (embedding_dim+ hidden_size)
        self.Wf = nn.Linear(concat_size,hidden_size)
        self.Wi = nn.Linear(concat_size,hidden_size)
        self.Wg = nn.Linear(concat_size,hidden_size)
        self.Wo = nn.Linear(concat_size,hidden_size)
        self.fc = nn.Linear(hidden_size,vocab_size)

    def forward(self, xs):
        # current xs shape is (batch_size,seqlen)
        xs = self.embedding(xs)
        # now shape is (batch_size,seqlen,embed_dim)
        xs = xs.transpose(0,1)
        # xs shape (seqlen,batch_size,embed_dim) or say (time,batch_size,embed_dim)
        seq_len = xs.size(0)
        batch_size = xs.size(1)

        h = torch.zeros(batch_size,self.hidden_size,device=xs.device)

        c = torch.zeros(batch_size,self.hidden_size,device=xs.device)

        outputs = []

        for t in range(seq_len): # now we will iterterate time wise 

            x = xs[t] # (batch_size,embed_dim)

            concat = torch.cat([h, x],dim=1) # (batch_size,hidden_state+embed_dim)
            f = torch.sigmoid(self.Wf(concat)) # (batch_size,hidden_state)
            i = torch.sigmoid(self.Wi(concat))  # (batch_size,hidden_state)
            g = torch.tanh(self.Wg(concat))  # (batch_size,hidden_state)
            o = torch.sigmoid(self.Wo(concat))  # (batch_size,hidden_state)
            c = f*c + i*g

            h = o*torch.tanh(c)  # (batch_size,hiddne_state)

            y = self.fc(h)  # (batch_size,vocab_size)

            outputs.append(y)

        return torch.stack(outputs) # (seq_len/time,batch_size,vocab_size)

In [77]:
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

model = LSTM_Torch(
    vocab_size=vocab_size,
    embedding_dim=32,
    hidden_size=128
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

X = X.to(device)
Y = Y.to(device)

batch_size = 64
epochs = 50

In [78]:
for epoch in range(epochs):

    total_loss = 0

    for i in range(0, len(X), batch_size):

        xb = X[i:i+batch_size]
        yb = Y[i:i+batch_size]

        logits = model(xb)

        logits = logits.permute(1,0,2)

        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            yb.reshape(-1)
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch: {epoch}, Batch Loss: {total_loss / (len(X)//batch_size)}\n",)
    if (epoch % 5 == 0):
        print(f"Test generation after {epoch} Epoch:\n{generate(model,device,"How do you convert between wealth and income tax?")}\n")

Epoch: 0, Batch Loss: 2.887048939346824

Test generation after 0 Epoch:
How do you convert between wealth and income tax? bitcooink fous ]inriS th.ve dakl iouy elseys vhiesytityriwrange 3fE. danomeafyb slios dN estocsaitrinuuiod the sa staeltp aw f,, vte qrotrmetidn sorty eugnt wywadkt an ,alimeuttthidvitharv. utetlipound ubpomsighindeotintrp theles awof fra ing rd toden bkeitha
ethpo a'yhusu ing, to rihit wh sodsr. son, ouelcevrae. thinnvts
ias tuisisriye s mr1f ad ireing isningd, oar.omito eritividelssrire acores inlej yhethoH c6f. teutl
i,e that Uot5alchimeitid eoz wh. roudvekerimhitatte satraucs the I aner Ict

Epoch: 1, Batch Loss: 2.4111670268955727

Epoch: 2, Batch Loss: 2.2069993338177194

Epoch: 3, Batch Loss: 2.0604371014137692

Epoch: 4, Batch Loss: 1.9487781019458983

Epoch: 5, Batch Loss: 1.8584074774639314

Test generation after 5 Epoch:
How do you convert between wealth and income tax? ersemle, izawrit is ite thecoutow is mat ferd. But an wor ghdet I'lding saych to meels

In [83]:
print(generate(model,device,"How do you convert between wealth and income tax? "))

How do you convert between wealth and income tax? If the extrauly clever lostic as change is not alread, whild use is not a rigaliscly more likely to be right. [1]

By wite objects to yever bad usually saches in fullo has goid agsounding inseenter Nonge ideas, whough, the fiever you would be truetome importanters. Nor justal to be a such a connection between there. I for this former riskely for sound of writing for my, be thine them puch in sang ton't this led more quesiplies modelen that important too.

In easion of woun write anologing ourtio
